# Models analyses

In [124]:
"""Import models."""
import pickle
from pathlib import Path

xgboost_model_path = Path("../models/xgboost_model.pkl")
random_forest_model_path = Path("../models/random_forest_model.pkl")

xgb_model = pickle.loads(xgboost_model_path.read_bytes())
rf_model = pickle.loads(random_forest_model_path.read_bytes())

print(f"Loaded XGBoost model from {xgboost_model_path}")
print(f"Loaded Random Forest model from {random_forest_model_path}")


Loaded XGBoost model from ../models/xgboost_model.pkl
Loaded Random Forest model from ../models/random_forest_model.pkl


In [125]:
"""Import test sets."""
import numpy as np
import pandas as pd
from IPython.display import display

TESTING_PATH = "../data/prepared/tennis_testing.xlsx"
RAW_TESTING_PATH = "../data/testing/tennis_testing.xlsx"

test_set = pd.read_excel(TESTING_PATH)
raw_test_df = pd.read_excel(RAW_TESTING_PATH)

target = "y"
y_test = test_set[target]

print(f"Loaded prepared test set with {len(test_set)} rows")
print(f"Loaded raw test set with {len(raw_test_df)} rows")


Loaded prepared test set with 16482 rows
Loaded raw test set with 16482 rows


### Baselines

In [126]:
def baseline_accuracy(name, pred_player1_wins, valid_mask, y_test):
    n_valid = valid_mask.sum()
    acc = (pred_player1_wins[valid_mask].astype(int) == y_test.loc[valid_mask]).mean()
    print(f"{name}: {acc:.2%}  (match validi: {n_valid}/{len(y_test)}, {n_valid/len(y_test):.1%})")
    return acc

maj_acc = y_test.value_counts(normalize=True).max()
print(f"Majority class: {maj_acc:.2%}  (match validi: {len(y_test)}/{len(y_test)}, 100.0%)")

valid_odds = test_set["Odds_1"].notna() & test_set["Odds_2"].notna() & (test_set["Odds_1"] != test_set["Odds_2"])
baseline_accuracy("Market (odds)", test_set["Odds_1"] < test_set["Odds_2"], valid_odds, y_test)

valid_rank = test_set["Rank_Diff"].notna()
baseline_accuracy("Ranking", test_set["Rank_Diff"] < 0, valid_rank, y_test)

valid_points = test_set["Points_Diff"].notna()
baseline_accuracy("Points", test_set["Points_Diff"] > 0, valid_points, y_test)

Majority class: 50.16%  (match validi: 16482/16482, 100.0%)
Market (odds): 68.51%  (match validi: 15997/16482, 97.1%)
Ranking: 63.65%  (match validi: 16451/16482, 99.8%)
Points: 63.38%  (match validi: 16451/16482, 99.8%)


np.float64(0.6338216521791988)

## Model Evaluation

In [127]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    log_loss,
    roc_auc_score,
)


def get_model_data(model, test_set):
    model_features = list(model.feature_names_in_)
    X = test_set[model_features]
    y = test_set["y"]
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return model_features, X, y, proba, pred


def evaluate_model(name, model, test_set, valid_odds):
    model_features, X, y, proba, pred = get_model_data(model, test_set)

    # valid_odds è una Series booleana allineata per indice a test_set/y -> serve come array posizionale
    odds_mask = valid_odds.loc[test_set.index].to_numpy()

    print(f"{name}")
    print("Accuracy (tutti i match):", f"{accuracy_score(y, pred):.2%}")
    print(
        "Accuracy (solo match con odds valide, confronto equo col mercato):",
        f"{accuracy_score(y[odds_mask], pred[odds_mask]):.2%}",
    )
    print("ROC AUC:", f"{roc_auc_score(y, proba):.4f}")
    print("Log loss:", f"{log_loss(y, proba):.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y, pred))

    return {
        "name": name,
        "features": model_features,
        "X": X,
        "y": y,
        "proba": proba,
        "pred": pred,
    }


xgb_results = evaluate_model("XGBoost", xgb_model, test_set, valid_odds)
rf_results = evaluate_model("Random Forest", rf_model, test_set, valid_odds)

agreement = (xgb_results["pred"] == rf_results["pred"]).mean()
prob_corr = pd.Series(xgb_results["proba"]).corr(pd.Series(rf_results["proba"]))

print(f"Model agreement: {agreement:.2%}")
print(f"Probability correlation: {prob_corr:.4f}")

XGBoost
Accuracy (tutti i match): 64.97%
Accuracy (solo match con odds valide, confronto equo col mercato): 65.36%
ROC AUC: 0.7160
Log loss: 0.6163
Confusion matrix:
[[5418 2850]
 [2924 5290]]
Random Forest
Accuracy (tutti i match): 64.88%
Accuracy (solo match con odds valide, confronto equo col mercato): 65.27%
ROC AUC: 0.7134
Log loss: 0.6179
Confusion matrix:
[[5442 2826]
 [2962 5252]]
Model agreement: 97.29%
Probability correlation: 0.9950


## Feature Usage

In [128]:
def source_feature_name(transformed_name, model_features):
    # Map transformed one-hot and missing-indicator names back to source columns.
    if transformed_name.startswith("num__missingindicator_"):
        return transformed_name.removeprefix("num__missingindicator_")
    if transformed_name.startswith("num__"):
        return transformed_name.removeprefix("num__")
    if transformed_name.startswith("cat__"):
        encoded = transformed_name.removeprefix("cat__")
    elif transformed_name.startswith("high_cardinality__"):
        encoded = transformed_name.removeprefix("high_cardinality__")
    else:
        return transformed_name

    for feature in model_features:
        if encoded.startswith(f"{feature}_"):
            return feature
    return encoded


def feature_usage(name, model, top_n=15):
    model_features = list(model.feature_names_in_)
    transformed_names = model[:-1].get_feature_names_out()
    estimator = model.named_steps["model"]

    if hasattr(estimator, "feature_importances_"):
        values = estimator.feature_importances_
        column = "Importance"
    else:
        values = np.abs(estimator.coef_[0])
        column = "Abs_Coefficient"

    usage = pd.DataFrame({"Feature": transformed_names, column: values})
    usage["Source_Feature"] = usage["Feature"].map(
        lambda feature: source_feature_name(feature, model_features)
    )

    usage_by_source = (
        usage.groupby("Source_Feature", as_index=False)[column]
        .sum()
        .sort_values(column, ascending=False)
    )

    print(f"{name} feature usage")
    display(usage_by_source.head(top_n))


feature_usage("XGBoost", xgb_model)
feature_usage("Random Forest", rf_model)


XGBoost feature usage


,Source_Feature,Importance
14,Surface_Elo_Diff,0.326118
2,Elo_Diff,0.227667
9,Points_Log_Ratio,0.107395
11,Rank_Log_Ratio,0.097140
10,Rank_Diff,0.062865
8,Points_Diff,0.044081
1,Dominance_Form_Diff,0.027306
0,Age_Diff,0.024637
6,Home_Diff,0.016497
4,H2H_Diff,0.015560


Random Forest feature usage


,Source_Feature,Importance
14,Surface_Elo_Diff,0.261039
2,Elo_Diff,0.182305
11,Rank_Log_Ratio,0.131525
9,Points_Log_Ratio,0.130706
8,Points_Diff,0.102565
10,Rank_Diff,0.073276
1,Dominance_Form_Diff,0.040844
13,Recent_Surface_Form_Diff,0.019067
0,Age_Diff,0.017815
12,Recent_Form_Diff,0.014882


## Most Confident Wrong Matches

In [129]:
def show_most_wrong(name, results, test_set, top_n=10):
    # Show confident errors: high confidence, wrong class.
    rows = pd.DataFrame({
        "Actual_Result": results["y"],
        "Predicted_Class": results["pred"].astype(int),
        "Prob_Player_1_Wins": results["proba"],
    }, index=test_set.index)

    rows["Actual_Winner"] = np.where(
        rows["Actual_Result"] == 1,
        test_set["Player_1"],
        test_set["Player_2"],
    )
    rows["Predicted_Winner"] = np.where(
        rows["Predicted_Class"] == 1,
        test_set["Player_1"],
        test_set["Player_2"],
    )
    rows["Confidence"] = np.where(
        rows["Predicted_Class"] == 1,
        rows["Prob_Player_1_Wins"],
        1 - rows["Prob_Player_1_Wins"],
    )

    detail_columns = [
        "Date", "Player_1", "Player_2", "Series", "Surface", "Round",
        "Odds_Diff", "Elo_Diff", "Surface_Elo_Diff", "Rank_Diff", "Points_Diff",
    ]
    detail_columns = [column for column in detail_columns if column in test_set.columns]

    wrong = rows[rows["Actual_Result"] != rows["Predicted_Class"]]
    wrong = wrong.sort_values("Confidence", ascending=False).head(top_n)

    print(f"{name}: most confident wrong matches")
    display(wrong.join(test_set[detail_columns]))


show_most_wrong("XGBoost", xgb_results, test_set)
show_most_wrong("Random Forest", rf_results, test_set)


XGBoost: most confident wrong matches


,Actual_Result,Predicted_Class,Prob_Player_1_Wins,Actual_Winner,Predicted_Winner,Confidence,Date,Player_1,Player_2,Series,Surface,Round,Odds_Diff,Elo_Diff,Surface_Elo_Diff,Rank_Diff,Points_Diff
10033,1,0,0.044253,Nardi L.,Djokovic N.,0.955747,2024-03-12,Nardi L.,Djokovic N.,Masters 1000,Hard,3rd Round,-2.803655,-618.660546,-595.256457,122.0,-9162.0
4589,1,0,0.046649,Vesely J.,Djokovic N.,0.953351,2022-02-24,Vesely J.,Djokovic N.,ATP500,Hard,Quarterfinals,-2.349105,-530.491373,-555.915824,122.0,-8325.0
7950,1,0,0.046886,Seyboth Wild T.,Medvedev D.,0.953114,2023-05-30,Seyboth Wild T.,Medvedev D.,Grand Slam,Clay,1st Round,-2.349105,-456.666800,-235.483806,170.0,-5981.0
15950,0,1,0.948524,Cerundolo J.M.,Sinner J.,0.948524,2026-05-28,Sinner J.,Cerundolo J.M.,Grand Slam,Clay,2nd Round,3.248146,543.267207,423.336648,-55.0,13815.0
14405,1,0,0.056540,Griekspoor T.,Sinner J.,0.943460,2025-10-05,Griekspoor T.,Sinner J.,Masters 1000,Hard,3rd Round,-2.924636,-333.894138,-348.549407,29.0,-9385.0
10545,1,0,0.060853,Tabilo A.,Djokovic N.,0.939147,2024-05-12,Tabilo A.,Djokovic N.,Masters 1000,Clay,3rd Round,-2.129566,-470.981685,-389.627019,31.0,-8670.0
4504,0,1,0.938772,Safiullin R.,Tsitsipas S.,0.938772,2022-02-18,Tsitsipas S.,Safiullin R.,ATP250,Hard,Quarterfinals,1.678431,366.533018,333.177476,-159.0,6963.0
12517,0,1,0.937228,Comesana F.,Zverev A.,0.937228,2025-02-21,Zverev A.,Comesana F.,ATP500,Clay,Quarterfinals,1.669542,399.993549,378.422920,-84.0,7462.0
4584,0,1,0.933288,Gojowczyk P.,Zverev A.,0.933288,2022-02-24,Zverev A.,Gojowczyk P.,ATP500,Hard,2nd Round,2.244316,338.075004,310.551667,-92.0,6810.0
7821,1,0,0.068234,Marozsan F.,Alcaraz C.,0.931766,2023-05-15,Marozsan F.,Alcaraz C.,Masters 1000,Clay,3rd Round,-3.125544,-421.102001,-363.380525,133.0,-6329.0


Random Forest: most confident wrong matches


,Actual_Result,Predicted_Class,Prob_Player_1_Wins,Actual_Winner,Predicted_Winner,Confidence,Date,Player_1,Player_2,Series,Surface,Round,Odds_Diff,Elo_Diff,Surface_Elo_Diff,Rank_Diff,Points_Diff
10033,1,0,0.032219,Nardi L.,Djokovic N.,0.967781,2024-03-12,Nardi L.,Djokovic N.,Masters 1000,Hard,3rd Round,-2.803655,-618.660546,-595.256457,122.0,-9162.0
4589,1,0,0.034395,Vesely J.,Djokovic N.,0.965605,2022-02-24,Vesely J.,Djokovic N.,ATP500,Hard,Quarterfinals,-2.349105,-530.491373,-555.915824,122.0,-8325.0
15950,0,1,0.958149,Cerundolo J.M.,Sinner J.,0.958149,2026-05-28,Sinner J.,Cerundolo J.M.,Grand Slam,Clay,2nd Round,3.248146,543.267207,423.336648,-55.0,13815.0
4504,0,1,0.955407,Safiullin R.,Tsitsipas S.,0.955407,2022-02-18,Tsitsipas S.,Safiullin R.,ATP250,Hard,Quarterfinals,1.678431,366.533018,333.177476,-159.0,6963.0
7950,1,0,0.045558,Seyboth Wild T.,Medvedev D.,0.954442,2023-05-30,Seyboth Wild T.,Medvedev D.,Grand Slam,Clay,1st Round,-2.349105,-456.666800,-235.483806,170.0,-5981.0
10545,1,0,0.051476,Tabilo A.,Djokovic N.,0.948524,2024-05-12,Tabilo A.,Djokovic N.,Masters 1000,Clay,3rd Round,-2.129566,-470.981685,-389.627019,31.0,-8670.0
9828,0,1,0.946817,Monteiro T.,Alcaraz C.,0.946817,2024-02-21,Alcaraz C.,Monteiro T.,ATP500,Clay,1st Round,2.244316,447.103710,367.246650,-115.0,8569.0
4584,0,1,0.944704,Gojowczyk P.,Zverev A.,0.944704,2022-02-24,Zverev A.,Gojowczyk P.,ATP500,Hard,2nd Round,2.244316,338.075004,310.551667,-92.0,6810.0
12517,0,1,0.942072,Comesana F.,Zverev A.,0.942072,2025-02-21,Zverev A.,Comesana F.,ATP500,Clay,Quarterfinals,1.669542,399.993549,378.422920,-84.0,7462.0
15151,1,0,0.061341,Kypson P.,De Minaur A.,0.938659,2026-02-24,Kypson P.,De Minaur A.,ATP500,Hard,1st Round,-2.244316,-428.116497,-403.487677,97.0,-3670.0
